In [1]:
import json
import os
import sys
from pathlib import Path
from typing import List

from pydantic import BaseModel

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "synthetic_data":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.config import load_config
from llm.build_llm import build_llm
from prompts.prompt_template import prompt as build_prompt
from utils.json_parser import parse_llm_json

from dotenv import load_dotenv

load_dotenv()

True

# Weather Data Generation

In [13]:
class WeatherData(BaseModel):
    date: int
    weather_summary: str


class MonthData(BaseModel):
    month: str
    weather_data: List[WeatherData]


output_schema = MonthData.model_json_schema()
schema_string = json.dumps(output_schema, indent=4)

In [14]:
role = """
You are a weather analyst creating concise daily weather summaries.
"""

task = """
Generate a weather summary for each day of the month for Colombo, Sri Lanka.
Month: {month}
Monthly description: {summary}
"""

context = """
The generated data will be used as synthetic input for hospital incident analysis.
Some incidents may be influenced by weather conditions, so the summaries should feel realistic.
"""

constraints = """
Return exactly one entry per day of the target month.
Keep each weather_summary to morning, afternoon and evening.
it is important to consider average rainy days when generating data as well so add occational rainy morning or  afternoon or evening based on average rainy days.
Use only the provided schema.
Do not use word "shower" it is not commonly use in sri lankan context to define rain.
"""

example = """
{
    \"month\": \"January\",
    \"weather_data\": [
        {
            \"date\": 1,
            \"weather_summary\": \"Mostly sunny with a brief afternoon rain.\"
        }
    ]
}
"""

In [15]:
BASE_DIR = Path.cwd()
if BASE_DIR.name != "synthetic_data":
    BASE_DIR = BASE_DIR / "synthetic_data"

PROJECT_ROOT = BASE_DIR.parent

INPUT_PATH = BASE_DIR / "resources" / "wether_summary.json"
OUT_PATH = BASE_DIR / "resources" / "generated_weather_data.json"

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
config["llm"]["default_provider"] = "openai"

openai_api_key = config["llm"]["providers"]["openai"]["api_key"]
if not openai_api_key or openai_api_key.startswith("${"):
    raise ValueError("Set OPENAI_API_KEY in your environment before running this notebook.")

openai_model = os.getenv("OPENAI_MODEL")
if openai_model:
    config["llm"]["providers"]["openai"]["model"] = openai_model

llm_router = build_llm(config)

with INPUT_PATH.open("r", encoding="utf-8") as file:
    monthly_summaries = json.load(file)

len(monthly_summaries)

12

In [16]:
generated_texts = []

for item in monthly_summaries:
    month = item["month"]
    summary = item["summery"]

    rendered_prompt = build_prompt(
        role=role,
        task=task.format(month=month, summary=summary),
        context=context,
        constraints=constraints,
        output_format=schema_string,
        examples=example,
    )

    generated_text = llm_router.generate_text(
        rendered_prompt,
        provider="openai",
        json_schema=output_schema,
        schema_name="weather_month_data",
    )

    generated_text = parse_llm_json(generated_text)

    generated_texts.append(generated_text)
    print(f"Generated text for {month}")




Generated text for January
Generated text for February
Generated text for March
Generated text for April
Generated text for May
Generated text for June
Generated text for July
Generated text for August
Generated text for September
Generated text for October
Generated text for November
Generated text for December


In [17]:
generated_text

{'month': 'December',
 'weather_data': [{'date': 1,
   'weather_summary': 'Morning: cloudy and humid. Afternoon: rain with thunder at times. Evening: cloudy breaks with a cooler breeze.'},
  {'date': 2,
   'weather_summary': 'Morning: crisp and sunny. Afternoon: warm with patchy cloud. Evening: mostly clear and cooler.'},
  {'date': 3,
   'weather_summary': 'Morning: light rain easing. Afternoon: brighter with sunny spells. Evening: mild with some clouds.'},
  {'date': 4,
   'weather_summary': 'Morning: sunny and crisp. Afternoon: warm, gentle breeze. Evening: clear and slightly cool.'},
  {'date': 5,
   'weather_summary': 'Morning: partly cloudy and sticky. Afternoon: brief rain, then clearing. Evening: drier with a cool breeze.'},
  {'date': 6,
   'weather_summary': 'Morning: sunny and fresh. Afternoon: warm and mostly clear. Evening: calm and cooler.'},
  {'date': 7,
   'weather_summary': 'Morning: hazy sun. Afternoon: warm with increasing cloud. Evening: rain later, turning cooler.

In [23]:
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(generated_texts, f, indent=4, ensure_ascii=False)

# Incident Data Generation

utput Format: Output the final dataset strictly as a JSON array of objects. Each object must contain the following keys:
   - "date": (string, YYYY-MM-DD)
   - "time": (string, HH:MM AM/PM)
   - "reporter_role": (string, e.g., "Senior Ward Nurse", "Attendant", "Medical Officer")
   - "incident_description": (string, the actual text written by the staff member)
   - "urgency": (string, "Low", "Medium", "High", "Critical")

In [7]:
from enum import Enum
from typing import List
from pydantic import BaseModel, Field, field_validator
import re

class UrgencyLevel(Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    CRITICAL = "Critical"

class IncidentReport(BaseModel):
    # Validates YYYY-MM-DD format
    date: str = Field(..., pattern=r"^\d{4}-\d{2}-\d{2}$")
    
    # Validates HH:MM AM/PM format
    time: str = Field(..., pattern=r"^(0[1-9]|1[0-2]):[0-5][0-9] (AM|PM)$")
    
    reporter_role: str = Field(..., min_length=1)
    incident_description: str = Field(..., min_length=5)
    urgency: UrgencyLevel

class IncidentDataset(BaseModel):
    reports: List[IncidentReport]

output_schema = IncidentDataset.model_json_schema()
schema_string = json.dumps(output_schema, indent=4)
print(schema_string)

{
    "$defs": {
        "IncidentReport": {
            "properties": {
                "date": {
                    "pattern": "^\\d{4}-\\d{2}-\\d{2}$",
                    "title": "Date",
                    "type": "string"
                },
                "time": {
                    "pattern": "^(0[1-9]|1[0-2]):[0-5][0-9] (AM|PM)$",
                    "title": "Time",
                    "type": "string"
                },
                "reporter_role": {
                    "minLength": 1,
                    "title": "Reporter Role",
                    "type": "string"
                },
                "incident_description": {
                    "minLength": 5,
                    "title": "Incident Description",
                    "type": "string"
                },
                "urgency": {
                    "$ref": "#/$defs/UrgencyLevel"
                }
            },
            "required": [
                "date",
                "time",
              

In [14]:
#prompt
role = """You are an expert synthetic data generator specializing in healthcare environments. You have extensive knowledge of hospital operations, incident reporting protocols, and the specific operational context of Sri Lankan healthcare facilities."""

task = """
    Generate a realistic dataset of exactly {num_incidents} incident reports submitted by various hospital staff members in a Sri Lankan hospital during the month of {month}. 

    All generated incidents must specifically fall under the purview of the {department_name} department. The nathure and responsibilities of this department are as follows:
    {department_nature}
    {department_responsibilities}

    I will provide the daily weather conditions for this month below. Use your analytical judgment to determine if and how the weather influenced specific incidents (e.g., heavy monsoon rains causing leaks or slip hazards, high humidity affecting equipment). Not all incidents should be weather-related; blend them naturally.
    Weather Data: {weather_data}
"""

constraints = """ 
    1. Temporal Distribution: Distribute the {num_incidents} unevenly across the month. Some days should have multiple incidents, while other days have zero.
    2. Persona Diversity: Use a diverse mix of tones, writing styles, vocabulary, and levels of urgency to reflect different hospital staff personalities (e.g., a stressed ward nurse, a formal hospital administrator, a hurried consultant, or support staff).
    3. Local Nuance: Subtly incorporate realistic Sri Lankan hospital terminology or contexts where appropriate (e.g., OPD, specific ward types, local syntax).
    4. Do not generate very long descriptions for incidents
"""

context = """ 
    The generated dataset will be used to train a machine learning model to automatically classify hospital incidents and route them to the correct department. Therefore, the incident descriptions must be highly realistic, nuanced, and distinctly require intervention from the target department without being overly simplistic or generic.
"""

In [2]:
BASE_DIR = Path.cwd()
if BASE_DIR.name != "synthetic_data":
    BASE_DIR = BASE_DIR / "synthetic_data"

PROJECT_ROOT = BASE_DIR.parent

WEATHER_DATA_PATH = BASE_DIR / "resources" / "generated_weather_data.json"
DEPARTMENTS_DATA_PATH = BASE_DIR / "resources" / "departments.json"
INCIDENTS_OUT_DIR = BASE_DIR / "resources" / "incidents"

with WEATHER_DATA_PATH.open("r", encoding="utf-8") as file:
    daily_weather_summaries = json.load(file)

print(f"weather data points: {len(daily_weather_summaries)}")

with DEPARTMENTS_DATA_PATH.open("r", encoding="utf-8") as file:
    department_summaries = json.load(file)

print(f"department data points: {len(department_summaries)}")


config = load_config(PROJECT_ROOT / "config" / "config.yaml")
config["llm"]["default_provider"] = "openai"

openai_api_key = config["llm"]["providers"]["openai"]["api_key"]
if not openai_api_key or openai_api_key.startswith("${"):
    raise ValueError("Set OPENAI_API_KEY in your environment before running this notebook.")

openai_model = os.getenv("OPENAI_MODEL")
if openai_model:
    config["llm"]["providers"]["openai"]["model"] = openai_model

llm_router = build_llm(config)


weather data points: 12
department data points: 12


In [ ]:
for department in department_summaries[:1]:
    for month_data in daily_weather_summaries[:1]: # Changed variable name for clarity
        rendered_prompt = build_prompt(
            role=role,
            task = task.format(
                num_incidents = 25,
                month = month_data["month"],
                department_name = department["department_name"],
                department_nature = department["nature"],
                department_responsibilities = department["responsibilities"],
                weather_data = month_data["weather_data"],
            ),
            context = context,
            constraints= constraints,
            output_format= schema_string,
            examples=""

            )
        generated_text = llm_router.generate_text(
            rendered_prompt,
            provider="openai",
        )

        generated_text = parse_llm_json(generated_text)

        generated_texts.append(generated_text)
        print(f"Generated text for {month_data["month"]}")
    print(f"Data generation done for department {department}")

Generated text for January
{
  "reports": [
    {
      "date": "2026-01-01",
      "time": "08:05 AM",
      "reporter_role": "OPD Nursing Officer",
      "incident_description": "New Year OPD rush: the token printer at main reception is misaligned, slowing registrations. Please switch to manual token chits and deploy an extra clerk to the OP ticket counter.",
      "urgency": "Medium"
    },
    {
      "date": "2026-01-03",
      "time": "09:10 AM",
      "reporter_role": "Security Supervisor",
      "incident_description": "Brief morning rain left the entrance damp; visitors are bunching at reception and blocking the OPD corridor. Request reception to open the side registration desk and direct foot traffic using the queue stand.",
      "urgency": "Medium"
    },
    {
      "date": "2026-01-05",
      "time": "02:15 PM",
      "reporter_role": "Dialysis Unit Medical Officer",
      "incident_description": "External calls for Dialysis are being routed to Dental via the main switchb

In [33]:

for department in department_summaries[10:12]:
    department_results = []  # collect all months for this department

    for month_data in daily_weather_summaries:
        rendered_prompt = build_prompt(
            role=role,
            task=task.format(
                num_incidents=25,
                month=month_data["month"],
                department_name=department["department_name"],
                department_nature=department["nature"],
                department_responsibilities=department["responsibilities"],
                weather_data=month_data["weather_data"],
            ),
            context=context,
            constraints=constraints,
            output_format=schema_string,
            examples=""
        )

        generated_text = llm_router.generate_text(
            rendered_prompt,
            provider="openai",
        )

        generated_text = parse_llm_json(generated_text)

        department_results.append({
            "month": month_data["month"],
            "data": generated_text
        })

        print(f"Generated text for {month_data['month']}")

    # create safe filename
    dept_name = department["department_name"].replace(" ", "_").lower()
    file_path = INCIDENTS_OUT_DIR / f"{dept_name}.json"

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(department_results, f, indent=2, ensure_ascii=False)

    print(f"Data generation done for department {department['department_name']}")
    print(f"Saved to {INCIDENTS_OUT_DIR}")

Generated text for January
Generated text for February
Generated text for March
Generated text for April
Generated text for May
Generated text for June
Generated text for July
Generated text for August
Generated text for September
Generated text for October
Generated text for November
Generated text for December
Data generation done for department Human Resources
Saved to /home/lakshan/ssp/IMS/synthetic_data/resources/incidents
Generated text for January
Generated text for February
Generated text for March
Generated text for April
Generated text for May
Generated text for June
Generated text for July
Generated text for August
Generated text for September
Generated text for October
Generated text for November
Generated text for December
Data generation done for department Quality Assurance department
Saved to /home/lakshan/ssp/IMS/synthetic_data/resources/incidents


In [3]:
TARGET_INCIDENT_YEAR = 2025

def normalize_incident_years(incidents_dir: Path, target_year: int = TARGET_INCIDENT_YEAR) -> None:
    for file_path in sorted(incidents_dir.glob("*.json")):
        with file_path.open("r", encoding="utf-8") as f:
            monthly_incidents = json.load(f)

        updated_reports = 0
        for month_entry in monthly_incidents:
            reports = month_entry.get("data", {}).get("reports", [])
            for report in reports:
                date_value = report.get("date")
                if isinstance(date_value, str) and len(date_value) == 10:
                    report["date"] = f"{target_year}-{date_value[5:]}"
                    updated_reports += 1

        with file_path.open("w", encoding="utf-8") as f:
            json.dump(monthly_incidents, f, indent=2, ensure_ascii=False)
            f.write("\n")

        print(f"Updated {updated_reports} incident dates in {file_path.name}")

normalize_incident_years(INCIDENTS_OUT_DIR)


Updated 300 incident dates in biomedical_engineering.json
Updated 300 incident dates in dietary_and_food_services.json
Updated 300 incident dates in facility_management.json
Updated 300 incident dates in finance_and_account.json
Updated 300 incident dates in housekeeping_department.json
Updated 300 incident dates in human_resources.json
Updated 300 incident dates in it.json
Updated 300 incident dates in quality_assurance_department.json
Updated 300 incident dates in reception.json
Updated 300 incident dates in security.json
Updated 300 incident dates in supply_department.json
Updated 300 incident dates in transport.json


In [10]:
import csv

CSV_OUT_DIR = BASE_DIR / "resources"
INCIDENTS_CSV_PATH = CSV_OUT_DIR / "incidents.csv"

department_name_by_file = {
    department["department_name"].replace(" ", "_").lower(): department["department_name"]
    for department in department_summaries
}

csv_rows = []

for file_path in sorted(INCIDENTS_OUT_DIR.glob("*.json")):
    assigned_department = department_name_by_file.get(
        file_path.stem,
        file_path.stem.replace("_", " ").title(),
    )

    with file_path.open("r", encoding="utf-8") as f:
        monthly_incidents = json.load(f)

    for month_entry in monthly_incidents:
        reports = month_entry.get("data", {}).get("reports", [])
        for report in reports:
            csv_rows.append({
                "date": report.get("date", ""),
                "time": report.get("time", ""),
                "reporter_role": report.get("reporter_role", ""),
                "assigned_department": assigned_department,
                "urgency": report.get("urgency", ""),
                "incident_description": report.get("incident_description", ""),
            })

csv_rows.sort(key=lambda row: (row["date"], row["time"], row["assigned_department"]))

with INCIDENTS_CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "date",
            "time",
            "reporter_role",
            "assigned_department",
            "urgency",
            "incident_description",
        ],
    )
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"Saved {len(csv_rows)} incidents to {INCIDENTS_CSV_PATH}")


Saved 3600 incidents to /home/lakshan/ssp/IMS/synthetic_data/resources/incidents.csv
